In [9]:
import os
import pandas as pd


**Loading Data Files**

In [10]:
DATA_DIR='./dataset' # Find the link to the dataset is in the README file

customers = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
order_payments = pd.read_csv(os.path.join(DATA_DIR, 'order_payments.csv'))
order_reviews = pd.read_csv(os.path.join(DATA_DIR, 'order_reviews.csv'))
products = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
sellers = pd.read_csv(os.path.join(DATA_DIR, 'sellers.csv'))
product_category_name_english = pd.read_csv(os.path.join(DATA_DIR, 'product_category_name_translation.csv'))

order_items = pd.read_csv(os.path.join(DATA_DIR, 'order_items.csv'),
                          parse_dates=[
                              'shipping_limit_date'
                          ])


orders = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'),
                     parse_dates=[
                          'order_purchase_timestamp',
                          'order_approved_at',
                          'order_delivered_carrier_date',
                          'order_delivered_customer_date',
                          'order_estimated_delivery_date'
    ])


**Checking Primary Keay Uniqueness**

In [ ]:
print("Unique order IDs:", orders['order_id'].nunique(), "out of", len(orders))
print("Unique customer IDs:", customers['customer_id'].nunique(), "out of", len(customers))
print("Unique sellers IDs:", sellers['seller_id'].nunique(),"out of", len(sellers))
print("Unique products IDs:", products['product_id'].nunique(),"out of", len(products))

print("Unique order_items IDs:", order_items['order_id'].nunique(),"out of", len(order_items))
print("Unique product categories:", product_category_name_english['product_category_name'].nunique(),"out of", len(product_category_name_english))
print("Unique payment types:", order_payments['payment_type'].nunique(),"out of", len(order_payments))

Unique order IDs: 99441 out of 99441
Unique customer IDs: 99441 out of 99441
Unique sellers IDs: 3095 out of 3095
Unique products IDs: 32951 out of 32951
Unique order_items IDs: 98666 out of 112650
Unique product categories: 71 out of 71
Unique payment types: 5 out of 103886


**Checking for Duplicate Values**

In [ ]:
print("Duplicate rows: ",orders.duplicated().sum())
print("Duplicate rows: ",order_items.duplicated().sum())
print("Duplicate rows: ",customers.duplicated().sum())
print("Duplicate rows: ",order_reviews.duplicated().sum())
print("Duplicate rows: ",products.duplicated().sum())
print("Duplicate rows: ",sellers.duplicated().sum())
print("Duplicate rows: ",order_payments.duplicated().sum())
print("Duplicate rows: ",product_category_name_english.duplicated().sum())



Duplicate rows:  0
Duplicate rows:  0
Duplicate rows:  0
Duplicate rows:  0
Duplicate rows:  0
Duplicate rows:  0
Duplicate rows:  0
Duplicate rows:  0


**Total Product Revenue**

In [ ]:
# Merge orders and order items on 'order_id'
order_merged_items = pd.merge(orders, order_items, on='order_id')

# Filter for delivered status
delivered_orders = order_merged_items[order_merged_items['order_status'] == 'delivered']

# Calculate Product  Revenue
total_revenue = (delivered_orders['price']).sum()

print(f"Total Revenue for Delivered Items: ${total_revenue:,.2f}")

Total Revenue for Delivered Items: $13,221,498.11


**Total Freight Cost**

In [ ]:
total_freight_value =delivered_orders['freight_value'].sum()
print(f"Total Freight Cost for Delivered Items: ${total_freight_value:,.2f}")

Total Freight Cost for Delivered Items: $2,198,275.64


**Total Inflow**

In [ ]:
total_inflow = total_revenue + total_freight_value
print(f"Total Inflow for Delivered Items: ${total_inflow:,.2f}")

Total Inflow for Delivered Items: $15,419,773.75


**Total Delivered Orders Count**

In [11]:
total_delivered_orders = (orders['order_status'] == 'delivered').sum()
print(f"Total Delivered Orders: {total_delivered_orders:,}")

Total Delivered Orders: 96,478


**Average Order Value**

In [ ]:
average_order_value = total_revenue / total_delivered_orders
print(f"Average Order Value: ${average_order_value:,.2f}")

Average Order Value: $137.04


**Average Delivery Days**

In [ ]:
average_delivery_days = (
    (delivered_orders['order_delivered_customer_date'] - delivered_orders['order_purchase_timestamp'])
    .dt.total_seconds() / 86400
).mean()

print(f"Average Delivery Days: {average_delivery_days:,.2f}")


Average Delivery Days: 12.56


**Average Review Score**

In [ ]:
review_merged_orders = order_reviews.merge(orders, on = 'order_id')
reviews_for_delivered_orders = review_merged_orders[review_merged_orders['order_status'] == 'delivered']

avg_review_score = reviews_for_delivered_orders['review_score'].mean()
print(f"Average Review Score: {avg_review_score:.1f} / 5.0")

Average Review Score: 4.2 / 5.0


**Late Delivery Rate**

In [ ]:
late_delivery_rate = (delivered_orders['order_delivered_customer_date'] > delivered_orders['order_estimated_delivery_date']).mean() * 100
print(f"Late Delivery Rate: {late_delivery_rate:.1f}%")

Late Delivery Rate: 8.1%


**On-time Delivery Rate**

In [ ]:
ontime_delivery_rate = 100 - late_delivery_rate
print(f"On-time Delivery Rate: {ontime_delivery_rate:.1f}%")

On-time Delivery Rate: 91.9%


**Total Breached Orders**

In [ ]:
total_breached_orders = (delivered_orders['order_delivered_customer_date'] > delivered_orders['order_estimated_delivery_date']).sum()
print(f"Total Breached Orders: {total_breached_orders:}")

Total Breached Orders: 7826
